In [6]:
import joblib
import pandas as pd
import numpy as np

# 1. Load the model
try:
    model = joblib.load('data/processed/seismic_model.pkl')
except FileNotFoundError:
    print("Error: File not found. Check your path.")

def unified_inference_engine(accel_data, temp_data, api_data, floor):
    """
    Processes sensors that match the model's training features.
    """
    # These 6 features must match your model's training exactly
    input_data = {
        'mag': api_data['mag'],
        'depth': api_data['depth'],
        'vibration_hz': accel_data['hz'],
        'acceleration_m2s': accel_data['m2s'],
        'floor_level': floor,
        'fire_sensor': 1.0 if temp_data['temp'] > 55 else 0.0
    }
    
    df = pd.DataFrame([input_data])
    
    # Run prediction
    prediction = model.predict(df)[0]
    confidence = np.max(model.predict_proba(df)) * 100
    
    labels = {0: "SAFE", 1: "WATCH", 2: "DANGER", 3: "EVACUATE"}
    
    return {
        "Status": labels[prediction],
        "Confidence": f"{confidence:.2f}%",
        "Action": execute_safety_actions(labels[prediction])
    }

def execute_safety_actions(status):
    if status == "EVACUATE":
        return "UNLOCK DOORS, SHUT OFF GAS, START ALARM"
    elif status == "DANGER":
        return "STOP ELEVATORS, ACTIVATE EMERGENCY LIGHTS"
    return "CONTINUE MONITORING"

# --- SIMULATION TEST ---
test_accel = {'hz': 8.5, 'm2s': 0.65}
test_temp  = {'temp': 62}
test_api   = {'mag': 6.8, 'depth': 25}

result = unified_inference_engine(test_accel, test_temp, test_api, floor=5)

print("="*40)
print(f"SYSTEM STATUS: {result['Status']}")
print(f"CONFIDENCE:    {result['Confidence']}")
print(f"ACTION:        {result['Action']}")
print("="*40)

SYSTEM STATUS: EVACUATE
CONFIDENCE:    70.00%
ACTION:        UNLOCK DOORS, SHUT OFF GAS, START ALARM
